In [5]:
import re
import nltk
from nltk.corpus import stopwords


import numpy as np
import pandas as pd

from keras.preprocessing.text import Tokenizer
from keras.utils import pad_sequences

from keras.layers import Conv1D, MaxPooling1D, Embedding, Concatenate, Dense, Input, Flatten
from keras.models import Model

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score

In [6]:
MAX_SEQUENCE_LENGTH = 100
MAX_NB_WORDS = 2000000
EMBEDDING_DIM = 100

In [7]:
df = pd.read_csv("Amazon_reviews.csv")

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 568454 entries, 0 to 568453
Data columns (total 10 columns):
 #   Column                  Non-Null Count   Dtype 
---  ------                  --------------   ----- 
 0   Id                      568454 non-null  int64 
 1   ProductId               568454 non-null  object
 2   UserId                  568454 non-null  object
 3   ProfileName             568438 non-null  object
 4   HelpfulnessNumerator    568454 non-null  int64 
 5   HelpfulnessDenominator  568454 non-null  int64 
 6   Score                   568454 non-null  int64 
 7   Time                    568454 non-null  int64 
 8   Summary                 568427 non-null  object
 9   Text                    568454 non-null  object
dtypes: int64(5), object(5)
memory usage: 43.4+ MB


In [9]:
data = df.sample(n=20000)

In [10]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 20000 entries, 107249 to 270650
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Id                      20000 non-null  int64 
 1   ProductId               20000 non-null  object
 2   UserId                  20000 non-null  object
 3   ProfileName             19999 non-null  object
 4   HelpfulnessNumerator    20000 non-null  int64 
 5   HelpfulnessDenominator  20000 non-null  int64 
 6   Score                   20000 non-null  int64 
 7   Time                    20000 non-null  int64 
 8   Summary                 19999 non-null  object
 9   Text                    20000 non-null  object
dtypes: int64(5), object(5)
memory usage: 1.7+ MB


In [11]:
data.reset_index(inplace=True, drop=True)

In [12]:
wl = nltk.WordNetLemmatizer()
ps = nltk.PorterStemmer()

In [13]:
def preprocess(text):
    text = text.lower()
    text = re.sub('[^a-zA-Z\s]','',text)
    stop = stopwords.words('english')
    text = [ps.stem(wl.lemmatize(word)) for word in text.split() if word not in stop]
    return text
    

In [15]:
processed_data = data['Text'].map(preprocess)

In [18]:
tokenizer = Tokenizer()#num_words = MAX_NB_WORDS)
tokenizer.fit_on_texts(processed_data)
sequences = tokenizer.texts_to_sequences(processed_data)

word_index = tokenizer.word_index
len(word_index)

28294

In [20]:
# dct = []
# for i in sequences:
#     dct.append(len(i))
        
# plt.hist(dct)
# plt.show()

In [23]:
final = pad_sequences(sequences, MAX_SEQUENCE_LENGTH)
labels = pd.get_dummies(data['Score'])
final.shape, labels.shape

((20000, 100), (20000, 5))

In [24]:
labels.head()

,1,2,3,4,5
0,0,0,0,1,0
1,0,0,0,0,1
2,0,0,0,0,1
3,0,0,0,0,1
4,0,0,0,1,0


In [25]:
x_train, x_test, y_train, y_test = train_test_split(final, labels, test_size = 0.2, random_state = 20)
x_test, x_val, y_test, y_val = train_test_split( final, labels, test_size=0.50, random_state=4)
x_train.shape, x_test.shape, x_val.shape, y_train.shape, y_test.shape, y_val.shape

((16000, 100), (10000, 100), (10000, 100), (16000, 5), (10000, 5), (10000, 5))

In [26]:
embedding_layer = Embedding(len(word_index) + 1,
                            EMBEDDING_DIM,
                            input_length=MAX_SEQUENCE_LENGTH)

In [27]:
convs = []
filter_sizes = [3,4,5]

sequence_input = Input(shape=(MAX_SEQUENCE_LENGTH,))
embedded_sequences = embedding_layer(sequence_input)

for fsz in filter_sizes:
    l_conv = Conv1D(128,fsz,activation='relu')(embedded_sequences)
    l_pool = MaxPooling1D(5)(l_conv)
    convs.append(l_pool)   
l_merge = Concatenate()(convs)
l_cov1= Conv1D(filters=128, kernel_size=5, activation='relu')(l_merge)
l_pool1 = MaxPooling1D(5)(l_cov1)
# l_cov2 = Conv1D(filters=128, kernel_size=5, activation='relu')(l_pool1)
# l_pool2 = MaxPooling1D(30)(l_cov2)
l_flat = Flatten()(l_pool1)
l_dense = Dense(128, activation='relu')(l_flat)
preds = Dense(5, activation='softmax')(l_dense)

model = Model(sequence_input, preds)
model.compile(loss='binary_crossentropy',
              optimizer='Nadam',
              metrics=['acc'])

model.summary()


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 100)]                0         []                            
                                                                                                  
 embedding (Embedding)       (None, 100, 100)             2829500   ['input_1[0][0]']             
                                                                                                  
 conv1d (Conv1D)             (None, 98, 128)              38528     ['embedding[0][0]']           
                                                                                                  
 conv1d_1 (Conv1D)           (None, 97, 128)              51328     ['embedding[0][0]']           
                                                                                              

In [28]:
history = model.fit(x_train, y_train, validation_data=(x_val, y_val),
          epochs=25, batch_size=150)

Epoch 1/25
107/107 [==============================] - 25s 208ms/step - loss: 0.3607 - acc: 0.6372 - val_loss: 0.3025 - val_acc: 0.6647
Epoch 2/25
106/107 [============================>.] - ETA: 0s - loss: 0.2885 - acc: 0.6787

KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline 
# list all data in history
print(history.history.keys())
# summarize history for accuracy
plt.plot(history.history['acc'])
plt.plot(history.history['val_acc'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()
# summarize history for loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [55]:
y_pred = model.predict(x_test, verbose=1)

313/313 [==============================] - 38s 114ms/step


In [62]:
def fx(array):
    return np.argmax(array)

temp = np.array(y_test)
st_pred = y_pred.tolist()
st_temp = temp.tolist()

arr = np.array(list(map(fx,st_temp)))
ans = np.array(list(map(fx,st_pred)))
f1 = f1_score(arr,ans, average='micro')
r1 = recall_score(arr, ans, average='micro')
p1 = precision_score(arr,ans, average='micro')
acc = accuracy_score(arr,ans)
print('f1',f1)
print('r1',r1)
print('p1',p1)
print('acc', acc)

f1 0.9148
r1 0.9148
p1 0.9148
acc 0.9148
